In [1]:
import os
import re
import fnmatch
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from procompa import get_project_root

PRJ_ROOT = get_project_root()
data_dir = PRJ_ROOT / "data"

In [2]:
TRIMMED_RUN = "22_YM_pae_trimmed"
CF_THRESHOLD = 80
results_csv_path = data_dir / "Pipeline" / TRIMMED_RUN / f"all_pdb_present_{TRIMMED_RUN}_pool_pipeline_complexes_combfold_results.csv"
yeastmap_tsv_path = data_dir / "Pipeline/12_Yeastmap/input_12_yeastmap.tsv"
ym_pdb_matches_path = data_dir / "YM_CP_complete_complex_pdb_mapping/all_YM_pdb_matches.csv"
MATCH_CLASS_ORDER = ["exact_homology_match", "superset", "partial", "no_sufficient_match", "no_homology_match"]

for required_path in (results_csv_path, yeastmap_tsv_path, ym_pdb_matches_path):
    assert required_path.exists(), f"missing: {required_path}"

cf_results = pl.read_csv(results_csv_path, infer_schema_length=0)
ym_input = pl.read_csv(yeastmap_tsv_path, separator="\t", infer_schema_length=0)
ym_pdb_matches = pl.read_csv(ym_pdb_matches_path, infer_schema_length=0)
assert cf_results["complex_ac"].is_unique().all(), "more than one results row per complex"
assert ym_input["#Complex ac"].is_unique().all(), "complex listed twice in the YeastMap tsv"
assert ym_pdb_matches["complex_ac"].is_unique().all(), "more than one PDB match row per complex"
assert set(ym_pdb_matches["match_class"]) <= set(MATCH_CLASS_ORDER), f"unknown match class: {set(ym_pdb_matches['match_class']) - set(MATCH_CLASS_ORDER)}"
missing_from_input = set(cf_results["complex_ac"]) - set(ym_input["#Complex ac"])
assert not missing_from_input, f"complexes in results but not in the YeastMap tsv: {sorted(missing_from_input)}"


def best_confidence(column: str) -> pl.Expr:
    """'{0: 78.5017, 1: 78.8618}' -> 78.8618; '{}' -> null"""
    return (pl.col(column).str.extract_all(r": -?[\d.]+")
            .list.eval(pl.element().str.slice(2).cast(pl.Float64)).list.max())


cf_candidates = (1, 2, 3, "true")   # true is 'not_run' for YeastMap complexes, kept so nothing is missed
cf_per_complex = cf_results.select(
    "complex_ac",
    pl.lit(True).alias("was_run"),
    (pl.sum_horizontal(*[pl.col(f"CF_{k}_n_assemblies").cast(pl.Int64) for k in cf_candidates]) > 0).alias("has_output"),
    pl.max_horizontal(*[best_confidence(f"CF_{k}_confidence") for k in cf_candidates]).alias("best_cf_confidence"),
)
assert (cf_per_complex["has_output"] == cf_per_complex["best_cf_confidence"].is_not_null()).all(), \
    "n_assemblies and confidence columns disagree"

complex_status = (
    ym_input.select(pl.col("#Complex ac").alias("complex_ac"))
    .join(ym_pdb_matches.select("complex_ac", "match_class"), on="complex_ac", how="left")
    .join(cf_per_complex, on="complex_ac", how="left")
    .with_columns(pl.col("was_run").fill_null(False), pl.col("has_output").fill_null(False))
)
missing_match_class = complex_status.filter(pl.col("match_class").is_null())["complex_ac"].to_list()
assert not missing_match_class, f"complexes without PDB match class: {missing_match_class}"
print("in tsv but not in the results csv (not run):",
      complex_status.filter(~pl.col("was_run")).select("complex_ac", "match_class").to_dicts())

in tsv but not in the results csv (not run): [{'complex_ac': 'pred_CPX_643', 'match_class': 'no_sufficient_match'}, {'complex_ac': 'pred_CPX_675', 'match_class': 'no_sufficient_match'}]


In [5]:
best_cf_column = f"n_best_cf_gt_{CF_THRESHOLD}"
is_best_cf_above = pl.col("best_cf_confidence") > CF_THRESHOLD
breakdown_aggregations = [
    pl.len().alias("n_input"),
    pl.col("was_run").sum().alias("n_run"),
    pl.col("has_output").sum().alias("n_with_output"),
    is_best_cf_above.sum().alias(best_cf_column),
    pl.col("complex_ac").filter(~pl.col("was_run")).sort().alias("complexes_not_run"),
    pl.col("complex_ac").filter(pl.col("was_run") & ~pl.col("has_output")).sort().alias("complexes_without_output"),
    pl.col("complex_ac").filter(is_best_cf_above).sort().alias(f"complexes_best_cf_gt_{CF_THRESHOLD}"),
]
breakdown_per_class = (
    complex_status.group_by("match_class").agg(breakdown_aggregations)
    .sort(pl.col("match_class").replace_strict({match_class: i for i, match_class in enumerate(MATCH_CLASS_ORDER)}))
)
breakdown_total = (complex_status.with_columns(pl.lit("total").alias("match_class"))
                   .group_by("match_class").agg(breakdown_aggregations))
cf_breakdown = pl.concat([breakdown_per_class, breakdown_total]).with_columns(
    (100 * pl.col("n_with_output") / pl.col("n_run")).round(1).alias("pct_output_of_run"),
    (100 * pl.col(best_cf_column) / pl.col("n_with_output")).round(1).alias(f"pct_gt_{CF_THRESHOLD}_of_output"),
)
with pl.Config(tbl_cols=-1, tbl_width_chars=200, fmt_table_cell_list_len=-1, fmt_str_lengths=1000):
    display(cf_breakdown)

match_class,n_input,n_run,n_with_output,n_best_cf_gt_80,complexes_not_run,complexes_without_output,complexes_best_cf_gt_80,pct_output_of_run,pct_gt_80_of_output
str,u32,u32,u32,u32,list[str],list[str],list[str],f64,f64
"""exact_homology_match""",2,2,2,1,[],[],"[""pred_CPX_523""]",100.0,50.0
"""superset""",2,2,2,2,[],[],"[""pred_CPX_29"", ""pred_CPX_521""]",100.0,100.0
"""partial""",27,27,19,6,[],"[""pred_CPX_362"", ""pred_CPX_386"", ""pred_CPX_426"", ""pred_CPX_436"", ""pred_CPX_486"", ""pred_CPX_501"", ""pred_CPX_565"", ""pred_CPX_677""]","[""pred_CPX_1"", ""pred_CPX_403"", ""pred_CPX_667"", ""pred_CPX_696"", ""pred_CPX_717"", ""pred_CPX_732""]",70.4,31.6
"""no_sufficient_match""",115,113,81,18,"[""pred_CPX_643"", ""pred_CPX_675""]","[""pred_CPX_17"", ""pred_CPX_183"", ""pred_CPX_185"", ""pred_CPX_188"", ""pred_CPX_190"", ""pred_CPX_366"", ""pred_CPX_370"", ""pred_CPX_391"", ""pred_CPX_396"", ""pred_CPX_428"", ""pred_CPX_441"", ""pred_CPX_454"", ""pred_CPX_498"", ""pred_CPX_545"", ""pred_CPX_553"", ""pred_CPX_555"", ""pred_CPX_561"", ""pred_CPX_576"", ""pred_CPX_586"", ""pred_CPX_618"", ""pred_CPX_623"", ""pred_CPX_626"", ""pred_CPX_639"", ""pred_CPX_650"", ""pred_CPX_651"", ""pred_CPX_673"", ""pred_CPX_676"", ""pred_CPX_690"", ""pred_CPX_692"", ""pred_CPX_693"", ""pred_CPX_704"", ""pred_CPX_718""]","[""pred_CPX_174"", ""pred_CPX_184"", ""pred_CPX_369"", ""pred_CPX_509"", ""pred_CPX_512"", ""pred_CPX_548"", ""pred_CPX_567"", ""pred_CPX_621"", ""pred_CPX_636"", ""pred_CPX_642"", ""pred_CPX_674"", ""pred_CPX_697"", ""pred_CPX_720"", ""pred_CPX_723"", ""pred_CPX_727"", ""pred_CPX_729"", ""pred_CPX_734"", ""pred_CPX_742""]",71.7,22.2
"""no_homology_match""",10,10,9,0,[],"[""pred_CPX_533""]",[],90.0,0.0
"""total""",156,154,113,27,"[""pred_CPX_643"", ""pred_CPX_675""]","[""pred_CPX_17"", ""pred_CPX_183"", ""pred_CPX_185"", ""pred_CPX_188"", ""pred_CPX_190"", ""pred_CPX_362"", ""pred_CPX_366"", ""pred_CPX_370"", ""pred_CPX_386"", ""pred_CPX_391"", ""pred_CPX_396"", ""pred_CPX_426"", ""pred_CPX_428"", ""pred_CPX_436"", ""pred_CPX_441"", ""pred_CPX_454"", ""pred_CPX_486"", ""pred_CPX_498"", ""pred_CPX_501"", ""pred_CPX_533"", ""pred_CPX_545"", ""pred_CPX_553"", ""pred_CPX_555"", ""pred_CPX_561"", ""pred_CPX_565"", ""pred_CPX_576"", ""pred_CPX_586"", ""pred_CPX_618"", ""pred_CPX_623"", ""pred_CPX_626"", ""pred_CPX_639"", ""pred_CPX_650"", ""pred_CPX_651"", ""pred_CPX_673"", ""pred_CPX_676"", ""pred_CPX_677"", ""pred_CPX_690"", ""pred_CPX_692"", ""pred_CPX_693"", ""pred_CPX_704"", ""pred_CPX_718""]","[""pred_CPX_1"", ""pred_CPX_174"", ""pred_CPX_184"", ""pred_CPX_29"", ""pred_CPX_369"", ""pred_CPX_403"", ""pred_CPX_509"", ""pred_CPX_512"", ""pred_CPX_521"", ""pred_CPX_523"", ""pred_CPX_548"", ""pred_CPX_567"", ""pred_CPX_621"", ""pred_CPX_636"", ""pred_CPX_642"", ""pred_CPX_667"", ""pred_CPX_674"", ""pred_CPX_696"", ""pred_CPX_697"", ""pred_CPX_717"", ""pred_CPX_720"", ""pred_CPX_723"", ""pred_CPX_727"", ""pred_CPX_729"", ""pred_CPX_732"", ""pred_CPX_734"", ""pred_CPX_742""]",73.4,23.9
